<a href="https://colab.research.google.com/github/Ilham-sy/psa-nlp-project/blob/main/NLP_GRP_PRJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PSA Translation Project

In [1]:
# Import libraries
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd


# Define the base path for files in Google Drive
drive_path = "/content/drive/MyDrive/"

multilanguage = pd.read_csv(f"{drive_path}Multilanguage_PSA.csv")
PSA = pd.read_csv(f"{drive_path}PSA_KE_Final.csv")

In [4]:
# Load the dataset
# This cell is redundant as data is loaded in StYVbsX22qZn
# multilanguage = pd.read_csv("Multilanguage_PSA.csv")
# PSA = pd.read_csv("PSA_KE_Final.csv")

In [5]:
# Check the dimensions
print("Multilanguage:", multilanguage.shape)
print("PSA:", PSA.shape)

Multilanguage: (6648, 9)
PSA: (2903, 8)


In [6]:
# Check the columns
print("Multilanguage columns:")
print(multilanguage.columns.tolist())

print("\nPSA columns:")
print(PSA.columns.tolist())

Multilanguage columns:
['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Source', 'Date', 'Metadata', 'PSA ID']

PSA columns:
['PSA_Id', 'Domain', 'Class', 'English', 'Kiswahili', 'Ekegusii', 'Dholuo', 'Somali']


In [7]:
# Drop unnecessary columns for multilanguage
multilanguage = multilanguage.drop(columns=["PSA ID", "Source", "Date", "Metadata"])

In [8]:
# Drop unnecessary columns for PSA
PSA = PSA.drop(columns=["Class", "Ekegusii", "Somali"])

In [9]:
# Rename columns
PSA = PSA.rename(columns={"PSA_Id": "PSA_ID"})

In [10]:
# Combine the datasets
combined = pd.concat([multilanguage, PSA], ignore_index=True)

In [11]:
# check the combined dataset
print(combined.shape)

combined.head()

(9551, 5)


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN


In [12]:
# Assign the combined DataFrame to 'df' for the cleaning process
df = combined

In [13]:
combined.to_csv(
    f"{drive_path}Combined_PSA_Raw.csv",
    index=False,
    encoding="utf-8-sig"
)

# Structural Cleaning

In [14]:
from google.colab import drive
import pandas as pd
import re

In [15]:
# Merge duplicate ID columns (if both exist)
if 'PSA ID' in df.columns and 'PSA_ID' in df.columns:
    df['PSA_ID'] = df['PSA_ID'].fillna(df['PSA ID'])
    df = df.drop(columns=['PSA ID'])

TEXT_COLS = ['English', 'Kiswahili', 'Dholuo']

def clean_text(val):
    if pd.isna(val):
        return val

    text = str(val)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove HTML entities
    text = re.sub(r'&nbsp;|&amp;|&quot;|&#\d+;', ' ', text)

    # Clean special character encoding
    text = re.sub(r'_x0092_', "'", text)   # right single quote
    text = re.sub(r'_x0093_', '"', text)   # left double quote
    text = re.sub(r'_x0094_', '"', text)   # right double quote
    text = re.sub(r'_x0096_', '-', text)   # en dash
    text = re.sub(r'_x0097_', '-', text)   # em dash
    text = re.sub(r'_x00[0-9A-Fa-f]{2}_', ' ', text)  # catch-all

    # Remove line breaks and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # Convert curly quotes to straight quotes
    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")

    # Remove extra spaces around brackets
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Collapse multiple spaces
    text = re.sub(r'\s{2,}', ' ', text)

    # Trim whitespace
    text = text.strip()

    return text

# Apply cleaning
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# Remove rows with empty English text
if 'English' in df.columns:
    df = df[df['English'].notna() & (df['English'].str.strip() != '')]

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()  # exact full-row duplicates (safe to keep)

# Remove content-duplicate rows (same PSA under a different PSA_ID)
dedup_cols = ['English', 'Kiswahili'] # Removed 'Source' and 'Date'
df = df.drop_duplicates(subset=dedup_cols, keep='first')

print(f"Duplicate rows removed: {before - len(df)}")

# Save cleaned dataset back to Google Drive
output_path = '/content/drive/MyDrive/PSA_Clean_v1.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Final rows: {len(df)}")
print(f"Saved as: {output_path}")

# Preview cleaned data
df.head()

Duplicate rows removed: 9
Final rows: 9540
Saved as: /content/drive/MyDrive/PSA_Clean_v1.csv


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN


In [18]:
df.shape

(9540, 5)

# Data Quality Cleaning

In [19]:
import pandas as pd
import numpy as np
import re

input_path = "/content/drive/MyDrive/PSA_Clean_v1.csv"

df = pd.read_csv(input_path)

print("Rows before cleaning:", len(df))
print("Columns:", list(df.columns))

Rows before cleaning: 9540
Columns: ['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo']


In [20]:
# Replace common missing value representations

missing_values = [
    "Unknown",
    "unknown",
    "UNKNOWN",
    "N/A",
    "n/a",
    "NA",
    "None",
    "null",
    ""
]

df = df.replace(missing_values, np.nan)

In [21]:
# Clean text column
TEXT_COLS = ["English","Kiswahili","Dholuo"]

In [22]:
def clean_text(text):

    if pd.isna(text):
        return text

    text = str(text)

    # remove emails
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)

    # remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # remove phone numbers
    text = re.sub(r'\+?\d[\d\-\s]{7,}\d', ' ', text)

    # remove website menus
    patterns = [

        r"Toggle navigation",
        r"Skip to main content",
        r"Home",
        r"Search website",
        r"Search",
        r"English",
        r"Français",
        r"Portuguese",
        r"About Us",
        r"About us",
        r"Contact us",
        r"Accessibility Statement",
        r"Privacy Policy",
        r"Cookie Policy",
        r"Terms of Use",
        r"Terms and Conditions",
        r"Copyright",

    ]

    for p in patterns:
        text = re.sub(p, " ", text, flags=re.IGNORECASE)

    # remove multiple spaces

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [23]:
for col in TEXT_COLS:

    if col in df.columns:

        df[col] = df[col].apply(clean_text)

In [24]:
# Remove empty English rows
df = df[
    df["English"].notna()
]

df = df[
    df["English"].str.strip() != ""
]

In [25]:
# Remove duplicate rows
before = len(df)

df = df.drop_duplicates()

print("Duplicates removed:", before-len(df))

Duplicates removed: 0


In [26]:
# Reset index
df = df.reset_index(drop=True)

In [27]:
# Data summary
print("="*50)

print("Rows:",len(df))

print("\nMissing values")

print(df.isnull().sum())

print("="*50)

Rows: 9540

Missing values
PSA_ID        128
Domain          0
English         0
Kiswahili       3
Dholuo       6650
dtype: int64


In [28]:
df[df["PSA_ID"].isna()].head(10)

,PSA_ID,Domain,English,Kiswahili,Dholuo
1483,NaN,Security,High traffic areas frequented by foreigners an...,Maeneo yenye msongamano mkubwa wa magari yanay...,NaN
1484,NaN,Security,Stay alert in locations frequented by tourists...,Kuwa macho katika maeneo yanayotembelewa na wa...,NaN
1485,NaN,Security,Review your personal security plans,Kagua mipango yako ya usalama wa kibinafsi,NaN
1486,NaN,Security,Be aware of your surroundings,Kuwa mwangalifu kuhusu mazingira yako,NaN
1487,NaN,Security,Monitor local media for updates,Fuatilia vyombo vya habari vya ndani kwa ajili...,NaN
1488,NaN,Security,Avoid protest areas and demonstrations,Epuka maeneo ya maandamano na maandamano,NaN
1489,NaN,Security,Avoid crowds,Epuka umati wa watu,NaN
1490,NaN,Security,Keep a low profile,Weka wasifu mdogo,NaN
1491,NaN,Security,Keep doors locked and windows rolled up while ...,Weka milango imefungwa na madirisha yamefungwa...,NaN
1492,NaN,Security,Notify friends and family of your whereabouts ...,Wajulishe marafiki na familia kuhusu mahali ul...,NaN


In [30]:
output_path="/content/drive/MyDrive/PSA_Clean_v2.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")

print(output_path)

Saved successfully!
/content/drive/MyDrive/PSA_Clean_v2.csv


In [31]:
print(df.head())

print(df.sample(10))

print(df.shape)

      PSA_ID  Domain                                            English  \
0  PSA000003  Health  PRESS RELEASE: JUNE 29, 2020 The EU through it...   
1  PSA000004  Health  This grant will be used by the WHO to support ...   
2  PSA000005  Health  Specifically, WHO Kenya will boost the respons...   
3  PSA000009  Health  Strengthening clinical care for high-consequen...   
4  PSA000010  Health  In recent years, investments in surveillance, ...   

                                           Kiswahili Dholuo  
0  TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...    NaN  
1  Ruzuku hii itatumiwa na WHO kuunga mkono juhud...    NaN  
2  Hasa, WHO Kenya itaongeza juhudi za kukabilian...    NaN  
3  Kuimarisha huduma ya kimatibabu kwa magonjwa y...    NaN  
4  Katika miaka ya hivi karibuni, uwekezaji katik...    NaN  
         PSA_ID           Domain  \
7011        480        Education   
7237        958        Education   
3436  PSA001826      Agriculture   
482   PSA001118           Health 